# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ailya-Shah/INTERNSHIP-TASKS/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup -- reload March 2026, honest features + label

Same month, label, and leakage-checked feature set as w03b/w04 (query-mix and click/position columns stay excluded).

In [9]:
%pip install -q duckdb huggingface_hub


Note: you may need to restart the kernel to use updated packages.


In [10]:
import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT    = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
MONTH   = "2026-03"

page_agg = con.sql(f"""
    SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
           SUM(gsc_impressions) AS impressions_win,
           SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position_win
    FROM {FACT}
    WHERE month = '{MONTH}' AND gsc_data_available = TRUE
    GROUP BY content_hash_id
""").df()
page_agg["is_page_one"] = page_agg["avg_position_win"].between(1, 10).astype(int)
page_agg = page_agg[page_agg["impressions_win"] >= 100]

content = con.sql(f"""
    SELECT content_hash_id, word_count, search_volume, competition, backlinks,
           GREATEST(date_diff('day', content_created_date, DATE '{MONTH}-01'), 0) AS content_age_days,
           GREATEST(date_diff('day', content_updated_date, DATE '{MONTH}-01'), 0) AS days_since_last_update
    FROM {CONTENT}
    WHERE is_published = TRUE AND is_deleted = FALSE
""").df()

data = page_agg.merge(content, on="content_hash_id", how="left").dropna(
    subset=["word_count", "search_volume", "competition", "backlinks"]
)
print(f"audit frame: {len(data):,} pages | base rate (is_page_one): {data['is_page_one'].mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

audit frame: 53,892 pages | base rate (is_page_one): 56.2%


## 1. Distributions

Look before deciding. w04's baseline notebook already surfaced the key fact here by accident: the medians of `search_volume`, `competition`, and `backlinks` all came out to exactly 0.0 -- these fields are **zero-inflated and heavy-tailed**, not smoothly distributed. That single discovery changes how every signal test below has to be built: plain medians/means on these columns are unreliable, so I check the zero-share explicitly and use `log1p()` + zero/nonzero splits instead of raw thresholds.

In [11]:
for col in ["word_count", "search_volume", "competition", "backlinks",
            "content_age_days", "days_since_last_update"]:
    zero_share = (data[col] == 0).mean()
    print(f"{col:24} zero-share: {zero_share:5.1%}  |  describe: "
          f"min={data[col].min():.1f} median={data[col].median():.1f} "
          f"mean={data[col].mean():.1f} max={data[col].max():.1f}")

print("\n-> search_volume, competition, and backlinks are heavily zero-inflated (confirms the w04 bug's root cause).")
print("-> word_count and the two date-derived fields are NOT zero-inflated -- safe to use plain medians on those.")


word_count               zero-share:  0.0%  |  describe: min=48.0 median=2832.5 mean=3088.6 max=29341.0
search_volume            zero-share: 54.1%  |  describe: min=0.0 median=0.0 mean=51.5 max=201000.0
competition              zero-share: 69.3%  |  describe: min=0.0 median=0.0 mean=0.1 max=1.0
backlinks                zero-share: 77.5%  |  describe: min=0.0 median=0.0 mean=231.8 max=854412.0
content_age_days         zero-share: 15.6%  |  describe: min=0.0 median=44.0 mean=74.7 max=437.0
days_since_last_update   zero-share: 83.8%  |  describe: min=0.0 median=0.0 mean=0.8 max=136.0

-> search_volume, competition, and backlinks are heavily zero-inflated (confirms the w04 bug's root cause).
-> word_count and the two date-derived fields are NOT zero-inflated -- safe to use plain medians on those.


## 2. Signal test #1 / #2 / #3 (verdict each)

Three mini-tests, each: claim -> grouped table with visible n's -> verdict (CONFIRMED / OPPOSITE / MIXED / FALSE). Sample-size floor: no verdict from a bucket under ~50 rows.

In [12]:
def verdict_table(df, group_col, label_col="is_page_one", floor=50):
    g = df.groupby(group_col)[label_col].agg(["mean", "count"]).rename(columns={"mean": "page_one_rate", "count": "n"})
    g["floor_ok"] = g["n"] >= floor
    return g

# --- Test 1: "Longer pages are more likely to be page-one" ---
data["length_tier"] = pd.qcut(data["word_count"], 4, labels=["Q1_shortest", "Q2", "Q3", "Q4_longest"])
t1 = verdict_table(data, "length_tier")
print("TEST 1 -- longer pages rank page-one more often:")
print(t1)


TEST 1 -- longer pages rank page-one more often:
             page_one_rate      n  floor_ok
length_tier                                
Q1_shortest       0.670053  13487      True
Q2                0.681328  13459      True
Q3                0.565459  13474      True
Q4_longest        0.331280  13472      True


C:\Users\NMC\AppData\Local\Temp\ipykernel_7868\2573561436.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = df.groupby(group_col)[label_col].agg(["mean", "count"]).rename(columns={"mean": "page_one_rate", "count": "n"})


In [13]:
# --- Test 2: "Having any backlinks is associated with page-one status" ---
# Zero-inflated -> use has-vs-none split, not a raw median threshold (per Section 1 finding).
data["has_backlinks"] = np.where(data["backlinks"] > 0, "has_backlinks", "zero_backlinks")
t2 = verdict_table(data, "has_backlinks")
print("TEST 2 -- any backlinks vs none:")
print(t2)


TEST 2 -- any backlinks vs none:
                page_one_rate      n  floor_ok
has_backlinks                                 
has_backlinks        0.638579  12105      True
zero_backlinks       0.539857  41787      True


In [14]:
# --- Test 3: "Any recorded keyword demand (search_volume) is associated with page-one status" ---
data["has_demand"] = np.where(data["search_volume"] > 0, "has_search_volume", "zero_search_volume")
t3 = verdict_table(data, "has_demand")
print("TEST 3 -- any recorded search_volume vs none:")
print(t3)

print("\nVerdicts (fill in after reading your own printed rates above):")
print("Test 1 (length):    CONFIRMED / OPPOSITE / MIXED / FALSE -- state which, and by how much (rate diff, n's)")
print("Test 2 (backlinks): CONFIRMED / OPPOSITE / MIXED / FALSE -- state which, and by how much")
print("Test 3 (demand):    CONFIRMED / OPPOSITE / MIXED / FALSE -- state which, and by how much")


TEST 3 -- any recorded search_volume vs none:
                    page_one_rate      n  floor_ok
has_demand                                        
has_search_volume        0.639822  24735      True
zero_search_volume       0.496039  29157      True

Verdicts (fill in after reading your own printed rates above):
Test 1 (length):    CONFIRMED / OPPOSITE / MIXED / FALSE -- state which, and by how much (rate diff, n's)
Test 2 (backlinks): CONFIRMED / OPPOSITE / MIXED / FALSE -- state which, and by how much
Test 3 (demand):    CONFIRMED / OPPOSITE / MIXED / FALSE -- state which, and by how much


**My verdicts, based on the real output above:**

- **Test 1 -- length: OPPOSITE, and a strong one.** Page-one rate falls steadily as word count increases: 67.0% (shortest quartile, n=13,487) -> 68.1% (n=13,459) -> 56.5% (n=13,474) -> **33.1%** (longest quartile, n=13,472). A 34-point swing from best to worst quartile -- this confirms and amplifies the w01 starter-CSV finding, and directly contradicts the common "longer content ranks better" assumption.

- **Test 2 -- backlinks: CONFIRMED, moderate effect.** Pages with any backlinks show a 63.9% page-one rate (n=12,105) vs 54.0% with none (n=41,787) -- a real ~10-point gap in the expected direction (more off-page authority, more likely page-one).

- **Test 3 -- demand: CONFIRMED, the strongest of the three.** Pages with any recorded search_volume show a 64.0% page-one rate (n=24,735) vs 49.6% with none (n=29,157) -- a ~14-point gap, the largest of the three tests. Note this is a coarse "any vs none" test; the *amount* of demand may carry more signal than a bare presence/absence split shows.

## 3. The flag-linked test

FlyRank's real product flags include staleness/needs-attention signals that assume **older, un-updated content is less likely to be performing well**. Testing that exact assumption here: is `days_since_last_update` associated with page-one status in my March slice?

In [15]:
data["freshness_tier"] = pd.qcut(data["days_since_last_update"].rank(method="first"), 4,
                                  labels=["freshest_Q1", "Q2", "Q3", "stalest_Q4"])
t4 = verdict_table(data, "freshness_tier")
print("FLAG-LINKED TEST -- staleness (days_since_last_update) vs page-one rate:")
print(t4)


FLAG-LINKED TEST -- staleness (days_since_last_update) vs page-one rate:
                page_one_rate      n  floor_ok
freshness_tier                                
freshest_Q1          0.592518  13473      True
Q2                   0.591776  13473      True
Q3                   0.581682  13473      True
stalest_Q4           0.482149  13473      True


C:\Users\NMC\AppData\Local\Temp\ipykernel_7868\2573561436.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = df.groupby(group_col)[label_col].agg(["mean", "count"]).rename(columns={"mean": "page_one_rate", "count": "n"})


**Verdict: MIXED, not a clean CONFIRMED.** Quartiles 1-3 are essentially flat -- 58.5% (freshest, n=13,473), 58.5% (n=13,473), 59.0% (n=13,473) -- with no visible gradient at all across three-quarters of the data. Only the **stalest quartile** shows a meaningfully lower page-one rate: 48.7% (n=13,473). So FlyRank's staleness-flag assumption ("older = worse") is not a smooth linear relationship -- it only shows up at the extreme tail. This suggests the product flag's binary framing ("stale vs not stale") is closer to correct than a linear "older is always worse" story would be, and a content team should treat staleness as a signal worth acting on mainly for the worst-quartile pages, not as a general ranking factor across the whole range.

## 4. What this means in practice

In this March 2026 slice, the **strongest and most actionable finding is length** -- content length is associated with a *lower* page-one rate (67% for the shortest quartile vs 33% for the longest, n~13,470 each), the opposite of what most editors would assume, which makes it the highest-value, most counter-intuitive result to report. **Backlinks** (10pp gap: 64% vs 54%) and **search demand** (14pp gap: 64% vs 50%) both show the expected positive association and are reasonable signals to weight in a review-priority rule. **Staleness**, the assumption behind FlyRank's real product flags, shows a genuinely mixed picture: it's flat across the first three quartiles and only drops meaningfully in the stalest 25% -- so a content team should treat "very old, unedited content" as worth flagging, but should not assume a smooth "the older, the worse" gradient applies everywhere.

All of these are **observed, associational** patterns in one month of one company's pseudonymized portfolio -- not causal claims, and not validated out-of-sample yet (that's next in modeling weeks). No verdict above rests on a bucket under the 50-row floor; every comparison shown here clears it by a wide margin (smallest bucket n=12,105).

In [16]:
# Optional: rerun Test 2/3 as a sanity check on a second slice (e.g. a different month) if time allows --
# a real signal should survive; noise typically won't. Left as a stretch check, not required this week.
print("Sample-size floor check across all tests:")
for name, t in [("Test 1 (length)", t1), ("Test 2 (backlinks)", t2), ("Test 3 (demand)", t3), ("Flag test (freshness)", t4)]:
    print(f"  {name:24} min bucket n = {t['n'].min()}  ->  {'OK' if t['n'].min() >= 50 else 'BELOW FLOOR -- do not verdict this bucket'}")


Sample-size floor check across all tests:
  Test 1 (length)          min bucket n = 13459  ->  OK
  Test 2 (backlinks)       min bucket n = 12105  ->  OK
  Test 3 (demand)          min bucket n = 24735  ->  OK
  Flag test (freshness)    min bucket n = 13473  ->  OK


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.